Amelia Polak-23362995
Mateusz Przywozny-23365021
Jakub Bolka- 23378247

A comment stating if the code executes to the end without an error.

A comment with a link to the third party source where you opted to reuse an existing implementation.
Basically any code you have resued 

COMMENT EVERY LINE IN YOUR/OUR  CODE 

STEP 1: LOAD AND PREPARE THE DATA (Mateusz):

Wrote code to download the dataset and split it into training, validation and test data.
The dataset is built into the torchvision framework.

Does the code run/work: YES

Sources (tutorials used for code): 
https://www.digitalocean.com/community/tutorials/vgg-from-scratch-pytorch
https://www.youtube.com/watch?v=ovZ_54IUSU4
https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html

In [ ]:
#import necessary libraries
import numpy as np #to work with arrays
import torch #deep learning framework
import torchvision.datasets as datasets #access to built in datasets, cifar10 is one of them
import torchvision.transforms as transform #to transform images
from torch.utils.data import Subset, DataLoader #to split data and load it later on

mean=[0.49139968, 0.48215827 ,0.44653124] #mean values for red,green and blue in cifar10
std=[0.24703233, 0.24348505, 0.26158768] #standard deviations for each colour
#values in range 0.0-1.0 (tensors)

#Transform image : VGG takes 32x32 images (dataset consists of 32x32 images),normalise for accuracy
#Split into two transforms(one for training data, other for the rest)
training_transform = transform.Compose([
    #transform.Resize((224,224)), #resize the image to 224x224

    transform.RandomHorizontalFlip(), #train on a flipped image
    transform.RandomRotation(20), #image may be slightly rotated
    transform.ToTensor(), #take image colours (0-255) and shrink the range to (0.0-1.0)-easier to read
    transform.Normalize(torch.Tensor(mean), torch.tensor(std)) #normalise to prevent the model from getting confused, eg when part of the image is super bright
])

#for validation and test data dont flip or rotate the images
test_transform = transform.Compose([
    #transform.Resize((224,224)),
    transform.ToTensor(),
    transform.Normalize(torch.Tensor(mean), torch.tensor(std))
])

#get data from the dataset (its split into training and test by default)
#root - name of the folder data are/will be in
#train-is data trained
#download = True ->download into the folder directly
training_full=datasets.CIFAR10(root = './dataset', train=True, download=True, transform=training_transform)
validation_full = datasets.CIFAR10(root = './dataset', train=True, download=True, transform=test_transform) #get the whole training data but dont modify as much as training data
test_set=datasets.CIFAR10(root = './dataset', train=False, download=True, transform=test_transform)

#create a list using permutations to select images at random, use random number for seed so it stays consistent
indices =torch.randperm(50000,generator=torch.Generator().manual_seed(10))

#split into training and validation, indices access the same images
training_set= Subset(training_full, indices[:45000])
validation_set = Subset(validation_full, indices[45000:])


#Load all 3 sets
#use batch size 32 as default
training_load = DataLoader(training_set, batch_size=32,shuffle=True) #shuffle training data for different order each epoch
validation_load = DataLoader(validation_set, batch_size=32)
test_load = DataLoader(test_set, batch_size=32)

100%|██████████| 170M/170M [24:30<00:00, 116kB/s]    


STEP 2:Build the CNN Layers(Amelia)

For this part I created a  VGG-11 Cnn using Pytorch.We chose VGG-11 instead of  using the full VGG-16 model as it has fewer convolutional layers.I added the convolutional layers ,ReLu activation functions ,max pooling and fully connected layers.

VGG-11 has:
* 8 convolutional layers -->learn features from images
* 3 fully connected layers -->used for classification
* 4096 neurons --> in first two fully connected layers 
* 5 max pooling layers --> reducing  the img size while keeping important features
* ReLU(activation function) -->helps the nn learn more complex patterns
* Dropout(0.5) --> helps reduce overfitting durng training
* Final layer has 10 outputs --> one for each CIFAR-10 class
* Images are resized to 32 x 32 --> to reduce training time 

Resources:
Reused code  + understanding the  VGG11 architecture :
* https://debuggercafe.com/implementing-vgg11-from-scratch-using-pytorch/
* https://www.kaggle.com/code/rightway11/vgg11tutorial

Understanding the VGG11 architecture:
* https://medium.com/@kfallon2010/ml-w-mnist-part-4-13c3e5d15c9f
* https://viso.ai/deep-learning/vgg-very-deep-convolutional-networks
* ChatGPT



Does the code work: Yes , I tested  the model using  random rgb image with size 224x224 .
* Input shape: (1,3,224,224)
* Output shape: (1,10)
* Parameters: 126,447,498


In [10]:
import torch.nn as nn #library for creating a nn layers 

class VGG11(nn.Module):
    
    def __init__(self,in_channels=3, num_classes=10):#3rgb and 10 cifar10 classes
        super().__init__()#initalization of parent class
        self.in_channels = in_channels # storing the  number of input channels 
        self.num_classes = num_classes #storing the number of classes 
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(self.in_channels,64,kernel_size=3,padding=1),#input channels to 64 output channels
            nn.ReLU(),#activation function
            nn.MaxPool2d(kernel_size=2,stride=2),#reducing img size by half
            
            nn.Conv2d(64,128,kernel_size=3,padding=1),#64 channels in , 128 out
            nn.ReLU(),#activation fucntion
            nn.MaxPool2d(kernel_size=2,stride=2),#reduce img size 
            
            nn.Conv2d(128,256,kernel_size=3,padding=1),#increasing channels to 256
            nn.ReLU(),#activation functino
            nn.Conv2d(256,256,kernel_size=3,padding=1),#keep 256 channels
            nn.ReLU(),#activation function
            nn.MaxPool2d(kernel_size=2,stride=2),#reduce img size 
            
            nn.Conv2d(256,512,kernel_size=3,padding=1),#increasing channels to 512
            nn.ReLU(),#activation functino
            nn.Conv2d(512,512,kernel_size=3,padding=1),#keep 512 channels
            nn.ReLU(),#activation function
            nn.MaxPool2d(kernel_size=2,stride=2),#reduce img size 
            
            nn.Conv2d(512,512,kernel_size=3,padding=1),#7th convolution layer
            nn.ReLU(),#activation function
            nn.Conv2d(512,512,kernel_size=3,padding=1),#last convolution - 8th
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2)# 5th final max pooling
            )    
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))#make the feature map 1x1 before classification
        
        #fully connected layers for classification
        self.linear_layers = nn.Sequential( 
            nn.Flatten(),#turns the feature maps into one vector
            nn.Linear(512,4096),#takes 25088 values and produces 4096 outputs
            nn.ReLU(),#activation function
            nn.Dropout(0.5),#ignoring the 50% of neurons during training in order to reduce the overfitting
            nn.Linear(4096,4096),#taking 4096 values and producing 4096 values
            nn.ReLU(),#activation fucntion
            nn.Dropout(0.5),#ignoring the 50% of neurons during training in order to reduce the overfitting
            nn.Linear(4096,self.num_classes)#produce 10 scores becasue CIRFAR10 has 10 classes 
                
        )
    #how img moves through the network
    def forward(self,x):#x is the input img
        x = self.conv_layers(x)#passing the img throught the convolution layers 
        x = self.avgpool(x)#making feature map 7 x 7
        x = self.linear_layers(x)#passing the img through connected layers 
        return x #return the 10 class scores
                
model = VGG11() #creating our VGG11 model
print("Model created successfully")#confirmation
        

Model created successfully


In [11]:
#Checking if our model works 

import torch #library for working with tensors 
#count the total parameters in our model
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

#creating one random RGB image with size 32x32
image_tensor = torch.randn(1, 3, 32, 32)

#testing model without training
model.eval()
with torch.no_grad():
    outputs = model(image_tensor)

#print the input and output shapes
print("Input shape:", image_tensor.shape)
print("Output shape:", outputs.shape)

Total parameters: 28,144,010
Input shape: torch.Size([1, 3, 32, 32])
Output shape: torch.Size([1, 10])
